In [9]:
import pandas as pd , plotly.express as px

In [10]:
# Load data 
df = pd.read_parquet('nyc_featured.parquet')
df

,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,...,open_data_channel_type,latitude,longitude,response_time_hours,response_time_days,complaint_day,complaint_hour,complaint_period,complaint_month,complaint_month_year
0,2023-06-01 01:04:22,2023-06-07 11:53:03,Department of Housing Preservation and Develop...,PLUMBING,BASIN/SINK,10035,EAST 118 STREET,Closed,The Department of Housing Preservation and Dev...,2023-06-07 00:00:00,...,PHONE,40.797065,-73.933855,154.811389,6.0,Thursday,1,Night,June,2023-06-01
1,2023-06-01 01:04:11,2023-06-01 01:36:43,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,2023-06-01 01:36:48,...,MOBILE,40.860566,-73.913341,0.542222,0.0,Thursday,1,Night,June,2023-06-01
2,2023-06-01 01:03:51,2023-06-01 01:35:53,New York City Police Department,Illegal Parking,Blocked Sidewalk,11230,OCEAN PARKWAY,Closed,The Police Department responded and upon arriv...,2023-06-01 01:35:58,...,PHONE,40.624388,-73.970522,0.533889,0.0,Thursday,1,Night,June,2023-06-01
3,2023-06-01 01:03:49,2023-06-01 01:10:17,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,2023-06-01 01:10:22,...,MOBILE,40.663251,-73.936208,0.107778,0.0,Thursday,1,Night,June,2023-06-01
4,2023-06-01 01:03:31,2023-06-01 01:12:32,New York City Police Department,Noise - Residential,Loud Music/Party,10009,AVENUE C,Closed,The Police Department responded to the complai...,2023-06-01 01:12:37,...,ONLINE,40.724409,-73.978610,0.150278,0.0,Thursday,1,Night,June,2023-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3129047,2022-06-01 01:08:15,2022-06-01 01:40:44,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10024,WEST 90 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:40:47,...,MOBILE,40.789428,-73.971178,0.541389,0.0,Wednesday,1,Night,June,2022-06-01
3129048,2022-06-01 01:07:59,2022-06-01 03:43:09,New York City Police Department,Noise - Commercial,Loud Talking,10466,WHITE PLAINS ROAD,Closed,The Police Department responded to the complai...,2022-06-01 03:43:13,...,MOBILE,40.896700,-73.855539,2.586111,0.0,Wednesday,1,Night,June,2022-06-01
3129049,2022-06-01 01:06:46,2022-06-02 11:00:32,New York City Police Department,Noise - Residential,Loud Music/Party,10466,EAST 231 STREET,Closed,The Police Department responded to the complai...,2022-06-02 11:01:13,...,MOBILE,40.892385,-73.859216,33.896111,1.0,Wednesday,1,Night,June,2022-06-01
3129050,2022-06-01 01:06:42,2022-06-01 01:20:07,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,11236,EAST 108 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:20:14,...,ONLINE,40.651047,-73.894917,0.223611,0.0,Wednesday,1,Night,June,2022-06-01


**Data Quality Note:**
A small number of records contain contradictory values — a closed_date 
exists but status remains 'Open'. This suggests either a system update 
lag or data entry inconsistency in the source data. These records were 
kept as-is since they represent a negligible portion of the dataset and 
removing them would not affect the overall analysis.

## 🗺️ Geographic Analysis
### Univariate


- What is the complaint count distribution across the 5 boroughs?


In [31]:
plot_df = df['borough'].value_counts().reset_index()
plot_df

,borough,count
0,BROOKLYN,882842
1,QUEENS,679385
2,BRONX,650416
3,MANHATTAN,599406
4,STATEN ISLAND,118866


In [32]:
px.bar(data_frame= plot_df , y='borough', x='count' , text_auto=True,
                   labels={'borough' : 'Borough Name' , 'count':'Number of Complaints'} , title='Complaint Distribution Across the 5 Boroughs' ,color_discrete_sequence= ["#3B5E64"])

- Which are the top 15 streets with the highest complaint volume?


In [33]:
plot_df = df['street_name'].value_counts().reset_index().head(15)
plot_df

,street_name,count
0,EAST 231 STREET,57348
1,BROADWAY,35999
2,GRAND CONCOURSE,18632
3,5 AVENUE,16718
4,7 AVENUE,13499
5,3 AVENUE,12814
6,8 AVENUE,12602
7,AMSTERDAM AVENUE,12409
8,PARSONS BOULEVARD,12090
9,ST NICHOLAS AVENUE,11216


In [34]:
px.bar(data_frame= plot_df, y='street_name', x='count'
        , labels={'street_name' : 'Street Name', 'count' :'Number of Complaints' } , text_auto=True, title='Top 15 streets with the highest complaint volume', color_discrete_sequence= ["#3B5E64"])

### Bivariate

- Which are the most ZIP codes that generate the most complaints within each borough?

In [35]:
plot_df = df.groupby(['borough', 'incident_zip']).size().reset_index(name='count')
plot_df = plot_df.sort_values(['borough', 'count'] , ascending=False)
plot_df = plot_df.groupby('borough').head(1)
plot_df

,borough,incident_zip,count
247,STATEN ISLAND,10314,18072
205,QUEENS,11385,39929
100,MANHATTAN,10031,32774
52,BROOKLYN,11226,46832
15,BRONX,10466,86854


In [38]:
px.bar(data_frame= plot_df, x='incident_zip', color='borough',y ='count'
        , labels={'incident_zip' : 'Incident ZIP Code', 'borough' :'borough Name' } , text_auto=True,
        title='The most ZIP codes that generate complaints within each borough?' ,color_discrete_sequence= ["#3B5E64" ,"#2E4F4D" ,"#6CA5A4" ,"#74BAB8" ,"#274C4B"])

- Which borough has the highest average response time in days?


In [39]:
plot_df= df.groupby('borough')['response_time_days'].mean().astype('int').reset_index().sort_values(by='response_time_days')
plot_df

,borough,response_time_days
0,BRONX,12
1,BROOKLYN,25
3,QUEENS,31
2,MANHATTAN,32
4,STATEN ISLAND,39


In [40]:
px.bar(data_frame=plot_df, x='borough', y='response_time_days', title='Average Response Time in Days',
                             labels={'borough':'Borough Name' , 'response_time_days' : 'Average response time'}, text_auto=True , color_discrete_sequence= ["#3B5E64"])

- What is the most 3 complaint type per borough?


In [41]:
plot_df = df.groupby(['borough', 'complaint_type']).size().reset_index(name='count')
plot_df = plot_df.sort_values(['borough' , 'count'] , ascending=False)
plot_df = plot_df.groupby('borough').head(3)
plot_df

,borough,complaint_type,count
772,STATEN ISLAND,Illegal Parking,11408
749,STATEN ISLAND,Electronics Waste Appointment,10077
800,STATEN ISLAND,Noise - Residential,8757
601,QUEENS,Illegal Parking,131472
540,QUEENS,Blocked Driveway,66360
629,QUEENS,Noise - Residential,65765
425,MANHATTAN,Illegal Parking,58960
453,MANHATTAN,Noise - Residential,58834
454,MANHATTAN,Noise - Street/Sidewalk,52112
248,BROOKLYN,Illegal Parking,159019


In [42]:
px.bar(plot_df, y='borough', x='count', color='complaint_type',
             barmode='group', 
             title='The most dominant complaint type per borough',
             labels={'borough': 'Borough Name', 'count': 'Number of complaints'},
             text_auto=True , color_discrete_sequence= ["#3B5E64" ,"#2E4F4D" ,"#6CA5A4" ,"#74BAB8" ,"#274C4B"])

- How does the reporting channel (Phone/Mobile/Online) differ across boroughs?

In [43]:
plot_df = df.groupby(['borough', 'open_data_channel_type']).size().reset_index(name='count')
plot_df = plot_df.sort_values(['borough' , 'count'] , ascending=False)
plot_df = plot_df.groupby('borough').head(3)
plot_df

,borough,open_data_channel_type,count
21,STATEN ISLAND,ONLINE,54648
23,STATEN ISLAND,PHONE,41815
20,STATEN ISLAND,MOBILE,11640
16,QUEENS,ONLINE,315418
18,QUEENS,PHONE,184969
15,QUEENS,MOBILE,137508
11,MANHATTAN,ONLINE,319885
13,MANHATTAN,PHONE,126453
10,MANHATTAN,MOBILE,125130
6,BROOKLYN,ONLINE,409130


In [44]:
px.bar(plot_df, y='borough', x='count', color='open_data_channel_type',
             barmode='group', 
             title='Reporting Method Distribution',
             labels={'borough': 'Borough Name', 'count': 'Usage Number'},
             text_auto=True , color_discrete_sequence= ["#3B5E64" ,"#2E4F4D" ,"#6CA5A4" ])

## ⏰ Temporal Analysis
### Univariate:



- How does complaint volume change month by month?


In [45]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

plot_df = df['complaint_month'].value_counts().reset_index(name='count')
plot_df['complaint_month'] = pd.Categorical(
    plot_df['complaint_month'], categories=month_order, ordered=True
)
plot_df = plot_df.sort_values('complaint_month').reset_index(drop=True)
plot_df


,complaint_month,count
0,January,225783
1,February,210375
2,March,235057
3,April,233515
4,May,267197
5,June,253934
6,July,276931
7,August,248483
8,September,254543
9,October,249559


In [46]:
px.line(plot_df, x='complaint_month', y='count',
        title='Monthly Complaint Volume Over Time',
        markers=True,
        template='plotly' )

- Which day of the week receives the most complaints?

In [103]:
plot_df = df['complaint_day'].value_counts().reset_index(name='count')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
plot_df['complaint_day'] = pd.Categorical(plot_df['complaint_day'], categories=day_order, ordered=True)
plot_df = plot_df.sort_values('complaint_day')
plot_df

,complaint_day,count
0,Monday,435641
1,Tuesday,433580
2,Wednesday,429493
4,Thursday,413767
3,Friday,416822
6,Saturday,399964
5,Sunday,401648


In [104]:
px.bar(plot_df , y='complaint_day', x='count' , title='Complaints by Day of the Week',
             text_auto=True, labels={'complaint_day' : 'Day' , 'count' :'Number of Complaints'} , color_discrete_sequence= ["#3B5E64"])

- What period of day dominates complaint submissions (Morning/Afternoon/Evening/Night)?

In [50]:
plot_df = df['complaint_period'].value_counts().reset_index(name='count')
plot_df

,complaint_period,count
0,Morning,826233
1,Night,784185
2,Afternoon,775388
3,Evening,545109


In [51]:
px.pie(data_frame= plot_df ,  names='complaint_period' , values='count' , title='Complaints by the Period of the day' , color_discrete_sequence= ["#3B5E64" ,"#2E4F4D" ,"#6CA5A4" ,"#74BAB8"])

### Bivariate:

- Which 3 complaint types peak in specific months ?


In [52]:
plot_df = df.groupby(['complaint_month', 'complaint_type']).size().reset_index(name='count')
plot_df= plot_df.sort_values(['complaint_month' , 'count'] , ascending=False)
plot_df= plot_df.groupby('complaint_month').head(3)
plot_df

,complaint_month,complaint_type,count
1920,September,Noise - Residential,44194
1891,September,Illegal Parking,35896
1921,September,Noise - Street/Sidewalk,18149
1728,October,Illegal Parking,36599
1756,October,Noise - Residential,27414
1719,October,HEAT/HOT WATER,25705
1565,November,Illegal Parking,34904
1557,November,HEAT/HOT WATER,33875
1592,November,Noise - Residential,21022
1397,May,Illegal Parking,42943


In [57]:
px.bar(plot_df , x='complaint_type', y='count', color='complaint_month',title='Top 3 complaints in each month',
              labels={'complaint_type':'Complaint Type', 'count':'Number of complaints' , 'complaint_month' :'Complaint Month'}
                ,color_discrete_sequence= ["#314954" ,"#305160" ,"#1E5772" ,"#74BAB8" ,"#274C4B" , "#6CA5A4" ,"#74BAB8" ,"#50A8A8" ,"#418AAE" ,"#2E4F4D" ,"#6CA5A4" ,"#0C312F" ,"#274C4B" ])

- How does complaint volume vary by hour across different boroughs?


In [58]:
plot_df = df.groupby(['borough', 'complaint_hour']).size().reset_index(name='count')
plot_df

,borough,complaint_hour,count
0,BRONX,0,27291
1,BRONX,1,19352
2,BRONX,2,13996
3,BRONX,3,10555
4,BRONX,4,10011
...,...,...,...
115,STATEN ISLAND,19,4639
116,STATEN ISLAND,20,4629
117,STATEN ISLAND,21,4971
118,STATEN ISLAND,22,4937


In [59]:
px.line(plot_df, x='complaint_hour', y='count',
        color='borough',
        title='Complaint Volume by Hour Across Boroughs',
        labels={'complaint_hour': 'Hour of Day', 'count': 'Number of Complaints'},
        markers=True,
        template='plotly')

* Complaints drop to their lowest point between 4-5 AM across all boroughs,
then rise sharply as the city wakes up, peaking around 10 AM.
Brooklyn consistently leads in complaint volume at every hour of the day.
Staten Island remains the quietest borough throughout.

- Which day + period combination has the highest complaint density? 

In [60]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
period_order = ['Morning', 'Afternoon', 'Evening', 'Night']

plot_df = df.groupby(['complaint_day', 'complaint_period']).size().reset_index(name='count')


pivot = plot_df.pivot(index='complaint_day', columns='complaint_period', values='count')

pivot = pivot.reindex(index=day_order, columns=period_order)
pivot

complaint_period,Morning,Afternoon,Evening,Night
complaint_day,,,,
Monday,128863,123142,80913,102723
Tuesday,137748,123920,80412,91500
Wednesday,132785,122089,82017,92602
Thursday,129289,115790,75186,93502
Friday,124074,112113,75682,104953
Saturday,90855,88868,73329,146912
Sunday,82619,89466,77570,151993


In [61]:
px.imshow(pivot,
          title='Complaint Density by Day and Period',
          labels={'x': 'Period of Day', 'y': 'Day of Week', 'color': 'Count'},
          aspect='auto',
          color_continuous_scale='Blues')

## 🏛️ Agency & Resolution Analysis
### Univariate:

- Which top 10 agency handles the most complaints overall?


In [62]:
plot_df = df['agency_name'].value_counts().reset_index().head(10)
plot_df

,agency_name,count
0,New York City Police Department,1368289
1,Department of Housing Preservation and Develop...,620139
2,Department of Sanitation,261900
3,Department of Environmental Protection,145900
4,Department of Parks and Recreation,117801
5,Department of Transportation,104566
6,Department of Buildings,94826
7,Department of Health and Mental Hygiene,82440
8,Economic Development Corporation,45559
9,Department of Homeless Services,32437


In [64]:
px.bar(plot_df , x='count', y='agency_name',title='Top 10 Agencies with the Highest Complaint Volume' , 
        labels={'count':'Number of Complaints' , 'agency_name':'Agency Name'}, color_discrete_sequence= ["#3B5E64"] , text_auto=True)


- What is the overall distribution of complaint status (Open/Closed/Pending)?

In [65]:
plot_df = df['status'].value_counts().reset_index(name='count')
plot_df

,status,count
0,Closed,2900053
1,In Progress,23659
2,Open,2102
3,Pending,1761
4,Unspecified,1324
5,Assigned,1300
6,Started,716


In [66]:
px.pie(plot_df, names='status', values='count', title='Distribution of Complaint Statuses')

* The pie chart above shows that the majority of complaints are closed, which is a positive sign that the city is addressing the issues raised by residents. However, there is still a significant portion of complaints that are open, indicating that there may be room for improvement in the city's response time and resolution process.

## Bivariate:

- Which agency has the fastest vs slowest average response time? 

In [67]:
plot_df = df.groupby('agency_name')['response_time_hours'].mean().reset_index().sort_values(by='response_time_hours', ascending=False)
plot_df

,agency_name,response_time_hours
7,Department of Parks and Recreation,6524.947180
10,Economic Development Corporation,6126.684055
4,Department of Health and Mental Hygiene,3801.462217
13,Taxi and Limousine Commission,1412.986098
0,Department of Buildings,1216.807119
2,Department of Education,1136.305243
6,Department of Housing Preservation and Develop...,472.447771
9,Department of Transportation,351.542850
12,Office of Technology and Innovation,345.295257
8,Department of Sanitation,328.130682


In [68]:
px.bar(plot_df, x='response_time_hours', y='agency_name',
       orientation='h',
       title='Average Response Time by Agency',
       labels={'response_time_hours': 'Average Response Time (Hours)', 
               'agency_name': 'Agency Name'},color_discrete_sequence= ["#3B5E64"])

* The Department of Parks and Recreation has the highest average response 
time (~6000 hours ≈ 250 days) because park maintenance complaints are 
low urgency and require scheduled work crews. The NYPD responds fastest 
because most police complaints require immediate on-ground response.

- Which 10 complaint types remain Open the longest without resolution? 


In [76]:
filtered_df = df[df['status'] == 'Open']

plot_df = filtered_df['complaint_type'].value_counts().reset_index(name='count').head(10)
plot_df

,complaint_type,count
0,General Construction/Plumbing,482
1,HEAT/HOT WATER,199
2,Building/Use,191
3,Graffiti,162
4,Hazardous Materials,141
5,PLUMBING,124
6,UNSANITARY CONDITION,121
7,Plumbing,85
8,GENERAL,73
9,PAINT/PLASTER,66


In [77]:
px.bar(plot_df , x='count' , y='complaint_type', title='Top 10 Open Complaints',
       labels={'count':'Number of Complaints' , 'complaint_type':'Complaint Type'}, color_discrete_sequence= ["#3B5E64"] , text_auto=True)

- What is the overall split between Phone, Mobile, Online, and other channels?


In [91]:
plot_df = df['open_data_channel_type'].value_counts().reset_index(name='count')
plot_df

,open_data_channel_type,count
0,ONLINE,1367924
1,PHONE,782457
2,MOBILE,621299
3,UNKNOWN,158878
4,OTHER,357


In [92]:
px.pie(plot_df, names='open_data_channel_type', values='count', title='Reporting Channel Distribution' , color_discrete_sequence= ["#3B5E64" ,"#2E4F4D" ,"#6CA5A4" ])

### Bivariate:

- Has mobile reporting grown over time while phone reporting declined?


In [99]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

plot_df = df.groupby(['complaint_month', 'open_data_channel_type']).size().reset_index(name='count')

plot_df = plot_df[plot_df['open_data_channel_type'].isin(['PHONE', 'MOBILE'])]
plot_df['complaint_month'] = pd.Categorical(
    plot_df['complaint_month'], categories=month_order, ordered=True
)
plot_df = plot_df.sort_values('complaint_month').reset_index(drop=True)
plot_df

,complaint_month,open_data_channel_type,count
0,January,MOBILE,42669
1,January,PHONE,56817
2,February,MOBILE,40536
3,February,PHONE,60312
4,March,PHONE,69757
5,March,MOBILE,45377
6,April,MOBILE,44986
7,April,PHONE,68478
8,May,PHONE,77243
9,May,MOBILE,55160


In [100]:
px.line(plot_df, x='complaint_month', y='count',
        color='open_data_channel_type',
        title='Mobile vs Phone Reporting Over Time',
        labels={'complaint_month': 'Month', 
                'count': 'Number of Complaints',
                'open_data_channel_type': 'Channel'},
        markers=True,
        template='plotly')

- Which borough is most digitally engaged (highest mobile/online ratio)?


In [55]:
filtered_df = df[(df['open_data_channel_type'] == 'MOBILE') | (df['open_data_channel_type'] == 'ONLINE')]
filtered_df

,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,...,open_data_channel_type,latitude,longitude,response_time_hours,response_time_days,complaint_day,complaint_hour,complaint_period,complaint_month,complaint_month_year
1,2023-06-01 01:04:11,2023-06-01 01:36:43,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,2023-06-01 01:36:48,...,MOBILE,40.860566,-73.913341,0.542222,0.0,Thursday,1,Night,June,2023-06-01
3,2023-06-01 01:03:49,2023-06-01 01:10:17,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,2023-06-01 01:10:22,...,MOBILE,40.663251,-73.936208,0.107778,0.0,Thursday,1,Night,June,2023-06-01
4,2023-06-01 01:03:31,2023-06-01 01:12:32,New York City Police Department,Noise - Residential,Loud Music/Party,10009,AVENUE C,Closed,The Police Department responded to the complai...,2023-06-01 01:12:37,...,ONLINE,40.724409,-73.978610,0.150278,0.0,Thursday,1,Night,June,2023-06-01
5,2023-06-01 01:02:55,2023-06-01 01:37:38,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,10040,SICKLES STREET,Closed,The Police Department responded to the complai...,2023-06-01 01:37:48,...,MOBILE,40.861328,-73.927757,0.578611,0.0,Thursday,1,Night,June,2023-06-01
6,2023-06-01 01:02:26,2023-06-01 03:57:25,New York City Police Department,Abandoned Vehicle,With License Plate,10469,PAULDING AVENUE,Closed,The Police Department responded to the complai...,2023-06-01 03:57:31,...,MOBILE,40.869872,-73.858541,2.916389,0.0,Thursday,1,Night,June,2023-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3129047,2022-06-01 01:08:15,2022-06-01 01:40:44,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10024,WEST 90 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:40:47,...,MOBILE,40.789428,-73.971178,0.541389,0.0,Wednesday,1,Night,June,2022-06-01
3129048,2022-06-01 01:07:59,2022-06-01 03:43:09,New York City Police Department,Noise - Commercial,Loud Talking,10466,WHITE PLAINS ROAD,Closed,The Police Department responded to the complai...,2022-06-01 03:43:13,...,MOBILE,40.896700,-73.855539,2.586111,0.0,Wednesday,1,Night,June,2022-06-01
3129049,2022-06-01 01:06:46,2022-06-02 11:00:32,New York City Police Department,Noise - Residential,Loud Music/Party,10466,EAST 231 STREET,Closed,The Police Department responded to the complai...,2022-06-02 11:01:13,...,MOBILE,40.892385,-73.859216,33.896111,1.0,Wednesday,1,Night,June,2022-06-01
3129050,2022-06-01 01:06:42,2022-06-01 01:20:07,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,11236,EAST 108 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:20:14,...,ONLINE,40.651047,-73.894917,0.223611,0.0,Wednesday,1,Night,June,2022-06-01


In [57]:
plot_df = filtered_df.groupby(['borough']).size().reset_index(name='count')
plot_df

,borough,count
0,BRONX,441083
1,BROOKLYN,583911
2,MANHATTAN,445015
3,QUEENS,452926
4,STATEN ISLAND,66288


In [61]:
px.bar(plot_df, y='borough', x='count', title='Complaint Volume by Borough for Mobile and Online Reports',
       labels={'borough': 'Borough Name', 'count': 'Number of Complaints'}, color_discrete_sequence=["#3B5E64" ] ,text_auto=True
       )

- Which complaint types are most reported via mobile vs phone?

In [11]:
filtered_df = df[(df['open_data_channel_type'] == 'MOBILE') | (df['open_data_channel_type'] == 'PHONE')]
filtered_df

,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,...,open_data_channel_type,latitude,longitude,response_time_hours,response_time_days,complaint_day,complaint_hour,complaint_period,complaint_month,complaint_month_year
0,2023-06-01 01:04:22,2023-06-07 11:53:03,Department of Housing Preservation and Develop...,PLUMBING,BASIN/SINK,10035,EAST 118 STREET,Closed,The Department of Housing Preservation and Dev...,2023-06-07 00:00:00,...,PHONE,40.797065,-73.933855,154.811389,6.0,Thursday,1,Night,June,2023-06-01
1,2023-06-01 01:04:11,2023-06-01 01:36:43,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,2023-06-01 01:36:48,...,MOBILE,40.860566,-73.913341,0.542222,0.0,Thursday,1,Night,June,2023-06-01
2,2023-06-01 01:03:51,2023-06-01 01:35:53,New York City Police Department,Illegal Parking,Blocked Sidewalk,11230,OCEAN PARKWAY,Closed,The Police Department responded and upon arriv...,2023-06-01 01:35:58,...,PHONE,40.624388,-73.970522,0.533889,0.0,Thursday,1,Night,June,2023-06-01
3,2023-06-01 01:03:49,2023-06-01 01:10:17,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,2023-06-01 01:10:22,...,MOBILE,40.663251,-73.936208,0.107778,0.0,Thursday,1,Night,June,2023-06-01
5,2023-06-01 01:02:55,2023-06-01 01:37:38,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,10040,SICKLES STREET,Closed,The Police Department responded to the complai...,2023-06-01 01:37:48,...,MOBILE,40.861328,-73.927757,0.578611,0.0,Thursday,1,Night,June,2023-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3129046,2022-06-01 01:08:22,2022-06-01 01:26:17,New York City Police Department,Noise - Commercial,Loud Talking,10128,EAST 92 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:26:15,...,MOBILE,40.781518,-73.948578,0.298611,0.0,Wednesday,1,Night,June,2022-06-01
3129047,2022-06-01 01:08:15,2022-06-01 01:40:44,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10024,WEST 90 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:40:47,...,MOBILE,40.789428,-73.971178,0.541389,0.0,Wednesday,1,Night,June,2022-06-01
3129048,2022-06-01 01:07:59,2022-06-01 03:43:09,New York City Police Department,Noise - Commercial,Loud Talking,10466,WHITE PLAINS ROAD,Closed,The Police Department responded to the complai...,2022-06-01 03:43:13,...,MOBILE,40.896700,-73.855539,2.586111,0.0,Wednesday,1,Night,June,2022-06-01
3129049,2022-06-01 01:06:46,2022-06-02 11:00:32,New York City Police Department,Noise - Residential,Loud Music/Party,10466,EAST 231 STREET,Closed,The Police Department responded to the complai...,2022-06-02 11:01:13,...,MOBILE,40.892385,-73.859216,33.896111,1.0,Wednesday,1,Night,June,2022-06-01


In [16]:
plot_df = filtered_df.groupby(['complaint_type', 'open_data_channel_type']).size().reset_index(name='count')
plot_df = plot_df.sort_values(['open_data_channel_type', 'count'], ascending=False)
plot_df = plot_df.groupby('open_data_channel_type').head(5)
plot_df

,complaint_type,open_data_channel_type,count
81,Illegal Parking,PHONE,69424
175,UNSANITARY CONDITION,PHONE,58377
14,Blocked Driveway,PHONE,55186
39,Derelict Vehicles,PHONE,45954
70,HEAT/HOT WATER,PHONE,44817
80,Illegal Parking,MOBILE,195211
114,Noise - Residential,MOBILE,111385
116,Noise - Street/Sidewalk,MOBILE,66637
13,Blocked Driveway,MOBILE,42451
69,HEAT/HOT WATER,MOBILE,39800


In [17]:
px.bar(plot_df, x='count', y='complaint_type', color='open_data_channel_type',
       barmode='group',
       title='Top Complaint Types by Reporting Channel (Mobile vs Phone)' , color_discrete_sequence= ["#85D2E0" ,"#2E4F4D" ],
       labels={'count':'Number of Complaints' , 'complaint_type':'Complaint Type' , 'open_data_channel_type':'Reporting Channel'})

## 📌 Complaint Type 

### Univariate:



- Complaint Distribution

In [70]:
plot_df = df['complaint_type'].value_counts().head(15).reset_index(name='count')

px.treemap(plot_df,
           path=['complaint_type'],
           values='count',
           title='Complaint Type Distribution — Treemap',
           color='count',
           color_continuous_scale='Blues')

- What are the top 10 most frequent complaint types across all of NYC?


In [49]:
plot_df = df['complaint_type'].value_counts().reset_index(name='count').head(10)
plot_df

,complaint_type,count
0,Illegal Parking,433372
1,Noise - Residential,348451
2,HEAT/HOT WATER,235037
3,Blocked Driveway,157831
4,Noise - Street/Sidewalk,151860
5,UNSANITARY CONDITION,95054
6,Abandoned Vehicle,63693
7,Noise - Commercial,62563
8,Missed Collection,59386
9,Noise - Vehicle,58331


In [51]:
px.bar(data_frame= plot_df , y='complaint_type', x='count' , text_auto=True, color_discrete_sequence= ["#3B5E64"] ,
       title='Top 10 Complaint Types occurring in NYC', labels={'complaint_type':'Complaint Type' , 'count':'Number of Complaints'})

- What is the distribution of complaint details within the top 5 complaint type?


In [108]:
top_types = df['complaint_type'].value_counts().head(5).index
plot_df = df[df['complaint_type'].isin(top_types)]
plot_df = plot_df.groupby(['complaint_type','complaint_detail']).size().reset_index(name='count')
plot_df = plot_df.sort_values('count', ascending=False).groupby('complaint_type').head(3)
plot_df

,complaint_type,complaint_detail,count
18,Noise - Residential,Loud Music/Party,231981
3,HEAT/HOT WATER,ENTIRE BUILDING,155222
6,Illegal Parking,Blocked Hydrant,127428
21,Noise - Street/Sidewalk,Loud Music/Party,125890
0,Blocked Driveway,No Access,117178
15,Illegal Parking,Posted Parking Sign Violation,99035
17,Noise - Residential,Banging/Pounding,90199
2,HEAT/HOT WATER,APARTMENT ONLY,79815
7,Illegal Parking,Blocked Sidewalk,47719
1,Blocked Driveway,Partial Access,40653


In [109]:
px.histogram(data_frame= plot_df , y='complaint_detail', x='count' , color='complaint_type',title='Complaint Types and Details Distribution',color_discrete_sequence=["#2E4F4D","#3B5E64" ,"#4CA5A4" ,"#298584" ,"#2F505D" , "#246261" ],
             labels={'complaint_detail':'Complaint Detail' , 'count':'Number of Complaints' , 'complaint_type':'Complaint Type'})

### Bivariate:



- What is the average response time per complaint type — which takes longest?

In [112]:
plot_df = df.groupby('complaint_type')['response_time_hours'].mean().round(2).reset_index().sort_values(by='response_time_hours', ascending=False).head(10)
plot_df

,complaint_type,response_time_hours
184,Window Guard,20018.26
161,Tattooing,18990.58
25,Calorie Labeling,18944.54
38,Day Care,18738.18
100,Mobile Food Vendor,18613.33
168,Trans Fat,18024.82
62,Food Establishment,13335.32
148,Smoking,12934.82
114,Non-Residential Heat,12558.56
104,New Tree Request,11932.06


In [113]:
px.bar(plot_df, x='response_time_hours', y='complaint_type',title='Average Response Time by Complaint Type',
       labels={'response_time_hours':'Average Response Time (Hours)','complaint_type':'Complaint Type'} ,color_discrete_sequence= ["#3B5E64"] , text_auto=True)

## Coordinates Location Map

In [71]:
sample = df.sample(5000)

px.scatter_map(sample,
                  lat='latitude',
                  lon='longitude',
                  color='borough',
                  hover_name='complaint_type',
                  zoom=10,
                  height=600,
                  title='NYC 311 Complaint Hotspots',
                  map_style='carto-positron')